In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import numpy as np
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer

In [2]:
USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project), Mozilla/5.0 (Windows NT 10.0; Win64; x64), Chrome/91.0.4472.124 Safari/537.36")

In [3]:
BASE = "https://www.spreewald-info.de"
headers = {"User-Agent": USER_AGENT}

# Links sammeln: Es gibt mehrere Seiten mit Bootsverleih, die alle unter "/paddeln/bootsverleih/" liegen.
url = BASE + "/paddeln/bootsverleih/"
html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

links = []

for a in soup.select("a"): #alle Links auf der Seite durchgehen
    href = a.get("href")
    if href.startswith("/paddeln/bootsverleih/") and href != "/paddeln/bootsverleih/":
        links.append(urljoin(BASE, href))
    # if href and "/paddeln/bootsverleih/" in href and href.count("/") >= 3:
    #     links.append(urljoin(BASE, href))

links = list(set(links)) #Duplikate entfernen

for i, link in enumerate(links, start=1):
     print(f"{i}. {link}")


1. https://www.spreewald-info.de/paddeln/bootsverleih/spreewald-kanus
2. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gasthof-zum-slawen
3. https://www.spreewald-info.de/paddeln/bootsverleih/spreehafen-burg
4. https://www.spreewald-info.de/paddeln/bootsverleih/bootsvermietung-henschelchen-schlepzig
5. https://www.spreewald-info.de/paddeln/bootsverleih/kleiner-spreewaldhafen
6. https://www.spreewald-info.de/paddeln/bootsverleih/bootshaus-leineweber
7. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-hafen-zur-alten-aalreuse
8. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gromsch
9. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-keutel
10. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-mahn
11. https://www.spreewald-info.de/paddeln/bootsverleih/dolzke-insel
12. https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-jentsch
13. https://www.spreewald-info.de/paddeln/bootsverleih/stand-up-paddlin

In [4]:
# Daten extrahieren
def parse_prices(url):

    html = requests.get(url, headers=headers).text
    soup = BeautifulSoup(html, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else "Unknown"

    prices = []

    for li in soup.select(".anbieterDetailsInfobox li"):
        text = li.get_text(" ", strip=True)

        if "Euro" in text or "€" in text:
            prices.append(text)

    return {
        "anbieter": title,
        "url": url,
        "preise": prices
    }

for i, link in enumerate(links, start=1):
    data = parse_prices(link)
    print(f"{i}. Anbieter: {data['anbieter']}")
    print(f"   URL: {data['url']}")
    print("   Preise:")
    for price in data["preise"]:
        print(f"     - {price}")


1. Anbieter: Bootsverleih Spreewald-Kanus
   URL: https://www.spreewald-info.de/paddeln/bootsverleih/spreewald-kanus
   Preise:
2. Anbieter: Gasthof zum Slawen - Bootsverleih in Raddusch
   URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gasthof-zum-slawen
   Preise:
3. Anbieter: Bootsverleih am Spreehafen Burg
   URL: https://www.spreewald-info.de/paddeln/bootsverleih/spreehafen-burg
   Preise:
4. Anbieter: Bootsvermietung Burkhard Henschelchen in Schlepzig
   URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootsvermietung-henschelchen-schlepzig
   Preise:
     - 1-er Paddelboot (Kajak) - bis 2 Stunden 20,00 Euro - 3 Stunden und Tagesmiete 25,00 Euro
     - 2-er Paddelboot (Kajak) - bis 2 Stunden 25,00 Euro - 3 Stunden und Tagesmiete 30,00 Euro
     - 3-er Paddelboot (Kajak) oder 3-er Kanadier - bis 2 Stunden 30,00 Euro - 3 Stunden und Tagesmiete 35,00 Euro
     - 4-er Kanadier - bis 2 Stunden 40,00 Euro - 3 Stunden und Tagesmietee 45,00 Euro
5. Anbieter: 

In [5]:
# Alle Anbieter und Preise sammeln. parce_prices() ist widerverwendbar
docs = []

for link in links:
    try:
        data = parse_prices(link)
        docs.append(data)

        print("\nAnbieter:", data["anbieter"])
        print("URL:", data["url"])

        for p in data["preise"]:
            print("  -", p)

    except Exception as e:
        print("Fehler bei:", link)
        print(e)


print("\nGesamt Anbieter gescraped:", len(docs))


Anbieter: Bootsverleih Spreewald-Kanus
URL: https://www.spreewald-info.de/paddeln/bootsverleih/spreewald-kanus

Anbieter: Gasthof zum Slawen - Bootsverleih in Raddusch
URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gasthof-zum-slawen

Anbieter: Bootsverleih am Spreehafen Burg
URL: https://www.spreewald-info.de/paddeln/bootsverleih/spreehafen-burg

Anbieter: Bootsvermietung Burkhard Henschelchen in Schlepzig
URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootsvermietung-henschelchen-schlepzig
  - 1-er Paddelboot (Kajak) - bis 2 Stunden 20,00 Euro - 3 Stunden und Tagesmiete 25,00 Euro
  - 2-er Paddelboot (Kajak) - bis 2 Stunden 25,00 Euro - 3 Stunden und Tagesmiete 30,00 Euro
  - 3-er Paddelboot (Kajak) oder 3-er Kanadier - bis 2 Stunden 30,00 Euro - 3 Stunden und Tagesmiete 35,00 Euro
  - 4-er Kanadier - bis 2 Stunden 40,00 Euro - 3 Stunden und Tagesmietee 45,00 Euro

Anbieter: Bootsverleih Kleiner Spreewaldhafen
URL: https://www.spreewald-info.de/paddeln

In [6]:
#JSON für RAG vorbereiten.

output_path = Path("providers.jsonl")

with output_path.open("w", encoding="utf-8") as f:
    for item in docs:
       #text feld für Embeddings vorbereiten
        text_parts = []
        if item.get("anbieter"):
            text_parts.append(item["anbieter"])
        if item.get("preise"):
            text_parts.append(" | ".join(item["preise"]))
        if item.get("url"):
            text_parts.append(item["url"])

        doc_entry = {
            "id": item.get("url"),
            "text": " — ".join(text_parts),       # zusammengeführter Text für Embeddings
            "metadata": item
        }
        f.write(json.dumps(docs, ensure_ascii=False) + "\n")
        if docs:
           print("\nAnbieter:", item["anbieter"])
           print("URL:", item["url"])
           print("Preise:")
           for p in item["preise"]:
               print("     -", p)



Anbieter: Bootsverleih Spreewald-Kanus
URL: https://www.spreewald-info.de/paddeln/bootsverleih/spreewald-kanus
Preise:

Anbieter: Gasthof zum Slawen - Bootsverleih in Raddusch
URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootsverleih-gasthof-zum-slawen
Preise:

Anbieter: Bootsverleih am Spreehafen Burg
URL: https://www.spreewald-info.de/paddeln/bootsverleih/spreehafen-burg
Preise:

Anbieter: Bootsvermietung Burkhard Henschelchen in Schlepzig
URL: https://www.spreewald-info.de/paddeln/bootsverleih/bootsvermietung-henschelchen-schlepzig
Preise:
     - 1-er Paddelboot (Kajak) - bis 2 Stunden 20,00 Euro - 3 Stunden und Tagesmiete 25,00 Euro
     - 2-er Paddelboot (Kajak) - bis 2 Stunden 25,00 Euro - 3 Stunden und Tagesmiete 30,00 Euro
     - 3-er Paddelboot (Kajak) oder 3-er Kanadier - bis 2 Stunden 30,00 Euro - 3 Stunden und Tagesmiete 35,00 Euro
     - 4-er Kanadier - bis 2 Stunden 40,00 Euro - 3 Stunden und Tagesmietee 45,00 Euro

Anbieter: Bootsverleih Kleiner Spreewaldhafe

In [7]:
#embedding
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

data =[]

with open("providers.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        data.append(item)
print(f"{len(data)} Dokumente geladen.")

17 Dokumente geladen.


In [8]:
texts = [docs['text'] for docs in data]

embeddings_preise = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings_preise).astype('float32')
print(f"Embeddings für {len(embeddings)} Dokumente erstellt.")

TypeError: list indices must be integers or slices, not str